# Lab 17: Prompt Chaining

Some problems can't be solved in a single prompt. For synthetic generation we used a two step process like this

1. Generate scenario combinations
2. Create inputs for each combination

In a similar way, sometimes we won't get good output with a single prompt and have to follow a step by step process. This is called *Prompt Chaining*.

Suppose after lot of attempts we are not able to get a prompt that is giving good output on the evaluation dataset, then we might decide to go with a three step process like this:

1. Analysis of JD with review comments (Input JD -> JSON review comments)
2. Rewrite specific sentences based on review comments (Input JD + review comments -> JSON rewritten sentences)
3. Merge the rewritten sentences with the original JD (Input JD + rewritten sentences -> Output JD)

Lets see how we can implement this

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="gpt-5.1-chat-latest")

## Step 1: Analyse the job description

First, we need to analyse the job description and get review comments on where it is lacking. Here are the system and user prompts for that

In [2]:
ANALYSIS_SYSTEM_PROMPT = """
You are an expert HR job description analyst specializing in inclusive hiring practices.

Analyze the provided job description for potential issues across these dimensions:

1. CLARITY: Identify sections with vague responsibilities, unclear expectations, or ambiguous requirements.
   Flag phrases like "various duties," "other tasks as assigned," or undefined acronyms.

2. JARGON: Flag unnecessarily technical language inappropriate for the role level.
   Consider whether terms would be understood by qualified candidates unfamiliar with internal terminology.

3. BIAS: Identify language that may discourage diverse candidates:
   - Gender-coded words (e.g., "rockstar," "ninja," "aggressive," "nurturing")
   - Age bias (e.g., "digital native," "recent graduate")
   - Exclusionary phrases (e.g., "culture fit," "work hard/play hard")
   - Excessive requirements (unnecessarily requiring degrees or years of experience)

4. MISSING INFORMATION: Note absent critical details:
   - Salary range or compensation structure
   - Work location/arrangement (remote/hybrid/onsite)
   - Reporting structure or team context
   - Clear distinction between required vs. preferred qualifications
   - Application process and timeline
   - Growth/development opportunities

5. SUMMARY: Provide 2-3 sentences describing overall quality and primary concerns.

For each issue you identify:
- Quote the exact problematic text
- Explain why it is problematic
- Suggest an improvement (when applicable)

Your output MUST be valid JSON that conforms exactly to the provided schema.
Do not include any text outside the JSON.

If information is missing, return empty arrays.
"""

ANALYSIS_USER_PROMPT = """
Analyze the following job description:

<job_description>
{job_description}
</job_description>

Return only JSON.
"""

Now we take the prompt and make a chain. Since we want structured output, we create a pydantic model that shows the structure of the required output. When we use `llm.with_structured_output`, it will automatically specify the output format in the API and parse the returned response as well.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel

class JDAnalysis(BaseModel):
    unclear_sections: list[str]
    jargon_terms: list[str]
    biased_language: list[str]
    missing_information: list[str]
    overall_summary: str

analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", ANALYSIS_SYSTEM_PROMPT),
    ("human", ANALYSIS_USER_PROMPT),
])
chain = analysis_prompt | llm.with_structured_output(JDAnalysis)

Let us test it out with a sample input

In [4]:
job_description = "We’re seeking a Forward Deployed Engineer. We want someone with 3+ years of software engineering experience with production systems. They should be rockstar programmers and problem solvers. They should have experience in a customer-facing technical role with a background in systems integration or professional services"
analysis = chain.invoke({"job_description": job_description})

In [5]:
print(analysis.unclear_sections)

['"Forward Deployed Engineer" (role title is not explained; responsibilities and scope are unclear)', 'The description lacks specific duties or expectations, making it unclear what the day‑to‑day work entails']


In [6]:
print(analysis.jargon_terms)

['"Forward Deployed Engineer" (may be internal terminology or unclear to external candidates)']


In [7]:
print(analysis.biased_language)

['"rockstar programmers" (gender‑coded, exclusionary, and vague)']


In [8]:
print(analysis.missing_information)

['Salary or compensation details', 'Work location or arrangement', 'Reporting structure', 'Clear required vs. preferred qualifications', 'Application process and timeline', 'Team context and scope of responsibilities', 'Growth or development opportunities']


## Step 2: Rewrite sentences based on review comments

Now we follow a similar process to ask the LLM to rewrite parts of the job description based on the review comments

In [9]:
REWRITE_SYSTEM_PROMPT = """
You are an expert HR editor specializing in rewriting job descriptions for clarity, inclusivity,
and accessibility.

You will receive:
1. The original job description.
2. A structured analysis of issues found in Step 1.

Your task is to rewrite ONLY the problematic sections, not the entire job description.

For each identified issue:
- Include the original problematic text (quoted exactly)
- Include the category (clarity, jargon, bias, or missing_information)
- Provide an improved, inclusive alternative that preserves meaning
- Maintain neutral, professional tone
- Ensure suggestions follow inclusive hiring practices

Return ONLY valid JSON matching the provided schema. Do not write any prose outside JSON.
"""

REWRITE_USER_PROMPT = """
Original Job Description:
-------------------------
{job_description}

Analysis Findings:
------------------
{analysis_json}

Rewrite ONLY the problematic sections using the schema.
Return only JSON.
"""

And here is the chain. In this prompt, we pass in the original job description as well as the JSON output from the first step. And the output from this step is also JSON, so we create the appropriate pydantic models

In [10]:
from typing import Literal

class RewrittenSection(BaseModel):
    category: Literal["clarity", "jargon", "bias", "missing_information"]
    original_text: str
    issue_explanation: str
    improved_text: str

class JDRewriteOutput(BaseModel):
    rewritten_sections: list[RewrittenSection]

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", REWRITE_SYSTEM_PROMPT),
    ("human", REWRITE_USER_PROMPT),
])
rewrite_chain = rewrite_prompt | llm.with_structured_output(JDRewriteOutput)
rewrite = rewrite_chain.invoke({"job_description": job_description, "analysis_json": analysis.model_dump_json()})

In [11]:
rewrite.rewritten_sections

[RewrittenSection(category='clarity', original_text='"Forward Deployed Engineer"', issue_explanation='The role title is unclear and may not convey responsibilities or scope to external candidates.', improved_text='Customer-Facing Software Engineer'),
 RewrittenSection(category='clarity', original_text='They should be rockstar programmers and problem solvers.', issue_explanation='The wording is vague and does not describe specific expectations or day‑to‑day responsibilities.', improved_text='They should be strong software engineers who can efficiently diagnose and resolve technical issues in production environments.'),
 RewrittenSection(category='jargon', original_text='"Forward Deployed Engineer"', issue_explanation='The term may be internal or unfamiliar to candidates, reducing accessibility.', improved_text='Engineer who works directly with customers to implement and support software solutions'),
 RewrittenSection(category='bias', original_text='They should be rockstar programmers an

## Step 3: Get the final Job description

Now we take the output from step 2 and merge it with the original job description

In [12]:
FINALISE_SYSTEM_PROMPT = """
You are an expert HR writer specializing in creating clear, concise, and inclusive job descriptions.

Your job is to produce the final polished version of the job description.

You will receive:
1. The original job description.
2. A list of rewritten sections (from Step 2).

Your tasks:
- Incorporate all improved rewritten sections into the original job description.
- Remove or replace the problematic text that was flagged in earlier steps.
- Maintain the original intent, structure, and role scope.
- Ensure clarity, inclusivity, and accessibility.
- Make tone consistent: professional, warm, and concise.
- Improve flow and readability where necessary.
- Do NOT invent new responsibilities, requirements, or benefits.

Return ONLY the final polished job description as plain text. Do not include JSON.
"""

FINALISE_USER_PROMPT = """
Original Job Description:
-------------------------
{job_description}

Rewritten Sections:
-------------------
{rewritten_sections_json}

Create the final polished job description by integrating the improvements.
Return only the final text.
"""

This time we want text output, so there is no pydantic model

In [13]:
finalise_prompt = ChatPromptTemplate.from_messages([
    ("system", FINALISE_SYSTEM_PROMPT),
    ("human", FINALISE_USER_PROMPT),
])
finalise_chain = finalise_prompt | llm
final_output = finalise_chain.invoke({
    "job_description": job_description, 
    "rewritten_sections_json": rewrite.model_dump_json()})

In [14]:
print(final_output.content)

Customer-Facing Software Engineer

We’re seeking a Customer-Facing Software Engineer with at least 3 years of experience working on production software systems. The ideal candidate has strong programming skills, enjoys solving complex technical problems, and has experience in customer-facing technical roles such as systems integration or professional services.

This role involves working directly with customers to understand technical needs, integrate software with existing systems, and support deployments in production environments. You will collaborate closely with cross-functional teams to ensure successful implementation of our solutions.

Key Responsibilities:
• Work directly with customers to implement and support software solutions.  
• Diagnose and resolve technical issues in production environments.  
• Integrate our software with customer systems and guide successful deployments.  
• Collaborate with internal teams to address customer needs and project goals.  

Required Qual

## Step 4: Evaluate

Once we have create this new version, we can run the evaluation dataset that we created before and check whether the new architecture is better or worse than the old one. Don't assume the new one is better! Sometimes the simpler architecture can perform better than the more complicated architecture, so we must always evaluate. This is why having the evaluation dataset is very important.